    <a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/Chapter_5/Ch5_Streamlit_Apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

    # Chapter 5: Data Apps with Streamlit (run from Colab)
    **Data Visualization — Ph.D. in Computing** · Dr. Ali Louati · IS Dept., PSAU

    ⏱️ Estimated lab time: **2 × 90 min**

    ### Learning objectives
    By the end of this lab you should be able to:
    1. Explain Streamlit's execution model (top-to-bottom re-run, caching, session state).
2. Write Streamlit apps from a notebook with `%%writefile` and **open them from Colab** through a public tunnel or Colab's port proxy.
3. Build a multi-filter dashboard with KPI cards, donut charts, rankings and drill-down tables (GRAALI case study).
4. Embed machine learning in an app **with sound methodology** (train/test split, leakage, model complexity, cluster validity).
5. Test apps headlessly with `streamlit.testing.AppTest` and deploy them to Streamlit Community Cloud.

    ### How to use this notebook
    * Run cells top to bottom with **Shift + Enter** (or *Runtime → Run all*).
    * Cells marked **✏️ Exercise** are for you. Cells marked **🔑 Solution** are collapsed: try first, then double-click to reveal.
    * Save your own copy: *File → Save a copy in Drive*.

## 0. Setup
Streamlit is not pre-installed on Colab.

In [ ]:
import sys, subprocess, os, time, re
IN_COLAB = "google.colab" in sys.modules
try:
    import streamlit
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)
    import streamlit
print("streamlit", streamlit.__version__, "| Colab:", IN_COLAB)

## 1. How Streamlit works
* Your script runs **top to bottom**. When a user touches a widget, Streamlit **re-runs the whole script** with the new widget value.
* No callbacks are needed (contrast with Dash, Ch. 4). The price: expensive work must be **cached** (`@st.cache_data`, `@st.cache_resource`) and anything that must survive re-runs goes in `st.session_state`.
* A Streamlit app is a **web server**, not a notebook cell. In Colab we therefore (1) write the app to a `.py` file, (2) start the server in the background, and (3) open it through a tunnel.

## 2. Helper: launch any app from Colab
Run this once. `launch("app.py")` starts the server and prints a link.
* **Method A — Cloudflare quick tunnel (recommended):** gives a public `https://….trycloudflare.com` URL. No account needed. Anyone with the link can see the app while the Colab runtime is alive, so don't use confidential data.
* **Method B — Colab port proxy:** keeps the app private to you; opens in a new tab. If the page stays blank, use Method A.

In [ ]:
CLOUDFLARED = "/usr/local/bin/cloudflared" if IN_COLAB else os.path.expanduser("~/cloudflared")
_procs = {}

def _get_cloudflared():
    if not os.path.exists(CLOUDFLARED):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED, url], check=True)
        os.chmod(CLOUDFLARED, 0o755)

def launch(app_file, port=8501, method="tunnel"):
    """Start `streamlit run app_file` in the background and return a URL to open it."""
    if port in _procs:                      # restart cleanly if already running on this port
        for p in _procs.pop(port): p.terminate()
    log = open(f"streamlit_{port}.log", "w")
    st_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", app_file, "--server.port", str(port),
                                "--server.headless", "true", "--server.enableCORS", "false",
                                "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
                               stdout=log, stderr=subprocess.STDOUT)
    _procs[port] = [st_proc]
    time.sleep(4)
    if method == "proxy" and IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(port, anchor_text=f"👉 Open {app_file}")
        return None
    if method == "tunnel":
        _get_cloudflared()
        cf = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        _procs[port].append(cf)
        for _ in range(60):
            line = cf.stdout.readline()
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
            if m:
                print(f"👉 {app_file} is live at: {m.group(0)}   (first load can take ~10 s)")
                return m.group(0)
        print("Tunnel did not start — try launch(..., method='proxy')")
    print(f"Running locally at http://localhost:{port}")

def stop_all():
    for procs in _procs.values():
        for p in procs: p.terminate()
    _procs.clear(); print("stopped")

## 3. First app: Iris explorer
`%%writefile` saves the cell to a file (it does **not** run it).

In [ ]:
%%writefile app_iris.py
import streamlit as st
import plotly.express as px

st.set_page_config(page_title="Iris Dashboard", layout="wide")
df = px.data.iris()                                   # built-in data: no files needed

st.title("Iris Explorer (Streamlit)")
st.write("Interactively explore feature distributions by species.")

with st.sidebar:                                       # controls
    st.header("Controls")
    feature = st.selectbox("Feature", ["sepal_length", "sepal_width", "petal_length", "petal_width"])
    bins = st.slider("Bins", 5, 60, 20)
    species = st.multiselect("Species", df["species"].unique().tolist(), default=df["species"].unique().tolist())

data = df[df["species"].isin(species)]                 # filter

c1, c2, c3 = st.columns(3)                             # KPI cards
c1.metric("Rows", len(data))
c2.metric(f"Mean {feature}", f"{data[feature].mean():.2f}" if len(data) else "–")
c3.metric("Species shown", len(species))

if data.empty:
    st.warning("Select at least one species.")
    st.stop()

fig = px.histogram(data, x=feature, color="species", nbins=bins, barmode="overlay", opacity=0.7)
st.plotly_chart(fig, width="stretch")

st.subheader("Preview (first 10 rows)")
st.dataframe(data.head(10))

In [ ]:
if IN_COLAB:
    launch("app_iris.py", port=8501)          # or: launch("app_iris.py", 8501, method="proxy")

**What to observe**
* Changing the feature re-runs the script and redraws the histogram.
* The bins slider changes the granularity of the distribution (recall Ch. 1 §4: your bin choice is part of the argument).
* De-selecting all species triggers `st.warning` + `st.stop()` — always handle empty filters.

## 4. Testing a Streamlit app without a browser
`streamlit.testing.v1.AppTest` runs the script headlessly and lets you set widgets and read outputs. This is how you would write CI tests for a research prototype — and how exercises can be auto-graded.

In [ ]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(os.path.abspath("app_iris.py")).run(timeout=30)
assert not at.exception, at.exception
assert at.title[0].value == "Iris Explorer (Streamlit)"
assert at.metric[0].value == "150"

at.multiselect[0].set_value(["setosa"]).run()
assert at.metric[0].value == "50"

at.multiselect[0].set_value([]).run()
assert at.warning[0].value == "Select at least one species."
print("✅ app_iris.py passes all tests")

## 5. Caching and session state
* `@st.cache_data` — memoises functions that **return data** (CSV loads, SQL queries, aggregations). Keyed on the arguments.
* `@st.cache_resource` — memoises **shared objects** (ML models, DB connections).
* `st.session_state` — per-user memory that survives re-runs (counters, multi-step forms, "model already trained" flags).

In [ ]:
%%writefile app_state.py
import time, streamlit as st, pandas as pd, numpy as np

@st.cache_data
def slow_load(n):
    time.sleep(2)                                   # pretend this is a slow query
    return pd.DataFrame(np.random.default_rng(0).normal(size=(n, 3)), columns=list("abc"))

n = st.slider("rows", 1_000, 50_000, 10_000, step=1_000)
t0 = time.time(); df = slow_load(n)
st.caption(f"loaded in {time.time() - t0:.2f} s (2 s the first time for each n, ~0 s after)")

if "clicks" not in st.session_state:
    st.session_state.clicks = 0
if st.button("Click me"):
    st.session_state.clicks += 1
st.metric("Button clicks this session", st.session_state.clicks)
st.line_chart(df.cumsum())

In [ ]:
at = AppTest.from_file(os.path.abspath("app_state.py")).run(timeout=30)
at.button[0].click().run(); at.button[0].click().run()
assert at.metric[0].value == "2"
print("✅ session state survives re-runs")

## 6. Case study: GRAALI — graduation-project management (CCES, PSAU)
GRAALI is a real Streamlit platform used to track GP1/GP2 graduation projects. Below is a teaching re-implementation on **synthetic data** with the same design patterns: sidebar drill-down filters, a headline KPI + KPI tiles, donut charts for composition, a supervisor-load ranking and a drill-down table with CSV export.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(2025)
n = 420
gp = pd.DataFrame({
    "project_id": [f"GP-{i:04d}" for i in range(n)],
    "department": rng.choice(["IS", "CS", "SE", "CE"], n, p=[.32, .30, .22, .16]),
    "gender": rng.choice(["Male", "Female"], n),
    "track": rng.choice(["Research", "Innovation", "Entrepreneurship"], n, p=[.45, .35, .20]),
    "type": rng.choice(["GP1", "GP2"], n),
    "academic_year": rng.choice(["2024-2025", "2025-2026"], n),
    "semester": rng.choice(["Semester 1", "Semester 2"], n),
    "supervisor": rng.choice([f"Dr. Supervisor {c}" for c in "ABCDEFGHIJKL"], n),
    "students": rng.integers(2, 5, n),
})
gp.to_csv("graali_synthetic.csv", index=False); gp.head()

In [ ]:
%%writefile app_graali.py
import pandas as pd, plotly.express as px, streamlit as st

st.set_page_config(page_title="GRAALI (teaching replica)", layout="wide")

@st.cache_data
def load(path="graali_synthetic.csv"):
    return pd.read_csv(path)

df = load()
st.title("GRAALI — Graduation Projects Dashboard")
st.caption("Teaching replica on synthetic data · CCES, PSAU")

with st.sidebar:
    st.header("Filters")
    year = st.selectbox("Academic year", sorted(df.academic_year.unique()), index=len(df.academic_year.unique()) - 1)  # smart default
    sem = st.multiselect("Semester", sorted(df.semester.unique()), default=sorted(df.semester.unique()))
    dept = st.multiselect("Department", sorted(df.department.unique()), default=sorted(df.department.unique()))
    gender = st.radio("Gender", ["All", "Male", "Female"], horizontal=True)
    track = st.multiselect("Track", sorted(df.track.unique()), default=sorted(df.track.unique()))
    gptype = st.multiselect("Project type", ["GP1", "GP2"], default=["GP1", "GP2"])

f = df[(df.academic_year == year) & df.semester.isin(sem) & df.department.isin(dept) & df.track.isin(track) & df.type.isin(gptype)]
if gender != "All":
    f = f[f.gender == gender]

st.metric("Total filtered projects", len(f))
k1, k2, k3, k4 = st.columns(4)
k1.metric("Semester 1", int((f.semester == "Semester 1").sum()))
k2.metric("Semester 2", int((f.semester == "Semester 2").sum()))
k3.metric("Students", int(f.students.sum()))
k4.metric("Supervisors", f.supervisor.nunique())

if f.empty:
    st.info("No projects match these filters.")
    st.stop()

d1, d2, d3 = st.columns(3)
for col, by, title in [(d1, "department", "Projects by department"), (d2, "track", "Projects by track"), (d3, "type", "Projects by type")]:
    col.plotly_chart(px.pie(f, names=by, hole=0.55, title=title), width="stretch")

load_ = f.groupby("supervisor").size().sort_values().rename("projects").reset_index()
st.plotly_chart(px.bar(load_, x="projects", y="supervisor", orientation="h", title="Supervision load"), width="stretch")

st.subheader("Projects (drill-down)")
st.dataframe(f, hide_index=True)
st.download_button("Download filtered CSV", f.to_csv(index=False), "graali_filtered.csv", "text/csv")

In [ ]:
at = AppTest.from_file(os.path.abspath("app_graali.py")).run(timeout=30)
assert not at.exception, at.exception
total_all = int(at.metric[0].value)
at.radio[0].set_value("Female").run()
assert int(at.metric[0].value) < total_all
print(f"✅ GRAALI replica OK — {total_all} projects in default view")
if IN_COLAB:
    launch("app_graali.py", port=8502)

> **Design critique (discuss).** Donuts are fine for 2–4 categories in an executive summary, but by Ch. 1's perception ranking a sorted bar chart is more accurate. Would you replace them? What would the dean lose or gain?

## 7. Streamlit + machine learning: Al-Kharj road-accident data (2022)
`2022.csv` holds **365 daily records for Al-Kharj in 2022**: car accidents, deaths, fatalities, injuries and injury accidents, each split into *within city* and *outside city* (`Date` in `dd/mm/yyyy`).
The cell below looks for the file locally, then in the course GitHub repository, then asks you to upload it. If none of that works, it generates synthetic data with the same columns so the lab still runs.

In [ ]:
DATA = "2022.csv"
DATA_URL = "https://raw.githubusercontent.com/alilouati/data-visualization-using-python/main/Chapter_5/2022.csv"
if not os.path.exists(DATA):
    try:
        pd.read_csv(DATA_URL, encoding="utf-8-sig").to_csv(DATA, index=False); print("downloaded from the course repository")
    except Exception:
        if IN_COLAB:
            print("Upload 2022.csv now (or press Cancel to use synthetic data).")
            try:
                from google.colab import files
                files.upload()
            except Exception as e:
                print("upload skipped:", e)

if not os.path.exists(DATA):
    days = pd.date_range("2022-01-01", "2022-12-31", freq="D")
    syn = {"Date": days.strftime("%d/%m/%Y"),
           "CarAccidentsOutsideCity": rng.poisson(2.7, len(days)), "CarAccidentsWithinCity": rng.poisson(15.7, len(days))}
    for where, lam_d, lam_i in [("OutsideCity", .1, .25), ("WithinCity", .07, .6)]:
        syn[f"Deceased{where}"] = rng.poisson(lam_d, len(days)); syn[f"Fatalities{where}"] = np.minimum(syn[f"Deceased{where}"], rng.poisson(lam_d, len(days)))
        syn[f"Injured{where}"] = rng.poisson(lam_i, len(days)); syn[f"AccidentsInjuries{where}"] = np.minimum(syn[f"Injured{where}"], rng.poisson(lam_i, len(days)))
    pd.DataFrame(syn).to_csv(DATA, index=False); print("⚠️ synthetic 2022.csv created (upload the real file for the real analysis)")

acc = pd.read_csv(DATA, encoding="utf-8-sig")
acc["Date"] = pd.to_datetime(acc["Date"], dayfirst=True)
print(acc.shape, acc.Date.min().date(), "→", acc.Date.max().date())
acc.describe().T.round(2)

### 7.1 Look before you model
Accident counts are small integers, many days have zero deaths, and there may be weekly and seasonal patterns. Check the weekly cycle and the within-city vs outside-city split first.

In [ ]:
import plotly.express as px
acc["weekday"] = acc.Date.dt.day_name()
order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
long = acc.melt(id_vars=["Date", "weekday"], value_vars=["CarAccidentsWithinCity", "CarAccidentsOutsideCity"], var_name="where", value_name="accidents")
px.box(long, x="weekday", y="accidents", color="where", category_orders={"weekday": order},
       title="Daily car accidents by weekday (Al-Kharj, 2022)", height=380).show()
weekly = acc.set_index("Date")[["CarAccidentsWithinCity", "CarAccidentsOutsideCity"]].rolling(7, center=True).mean().reset_index()
px.line(weekly, x="Date", y=["CarAccidentsWithinCity", "CarAccidentsOutsideCity"], title="7-day rolling mean", height=330).show()

### 7.2 Methodology first (PhD level)
The original app created the label *Severe / Not severe* from one column (above its mean) and trained a full-depth decision tree on the **other** columns, evaluated on the **training data**. Three problems:
1. **No hold-out set** → training accuracy of an unpruned tree is close to 100 % and says nothing about generalisation.
2. **Target leakage via proxies** → some predictors are near-copies of the labelling column. In this dataset `AccidentsInjuriesWithinCity` (accidents with injuries) and `InjuredWithinCity` (people injured) are strongly correlated, so using one to predict the other is trivial; the tree learns the definition, not the phenomenon.
3. **Unbounded complexity** → an unpruned tree is unreadable (the original needed a 100 × 71-inch figure).

The improved app adds a stratified train/test split, a depth control, a majority-class baseline, a confusion matrix, feature importances, a **leakage check**, optional calendar features, and silhouette scores to choose *k* for K-Means.

In [ ]:
%%writefile app_alkharj.py
import numpy as np, pandas as pd, plotly.express as px, streamlit as st, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

st.set_page_config(page_title="Al-Kharj Road Accidents 2022", layout="wide")
WEEK = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

@st.cache_data
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")          # the file starts with a byte-order mark
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    return df

df = load_csv("2022.csv")
num = df.select_dtypes("number").columns.tolist()
if len(num) < 2:
    st.error("The dataset must contain at least two numeric columns."); st.stop()

st.title("Al-Kharj Road Accidents Dashboard (2022)")
k1, k2, k3, k4 = st.columns(4)
k1.metric("Days", len(df))
k2.metric("Car accidents (within city)", int(df.get("CarAccidentsWithinCity", df[num[0]]).sum()))
k3.metric("Car accidents (outside city)", int(df.get("CarAccidentsOutsideCity", df[num[0]]).sum()))
dead = [c for c in num if c.startswith("Deceased")]
k4.metric("Deceased", int(df[dead].sum().sum()) if dead else "–")

tab1, tab2, tab3 = st.tabs(["📈 Explore", "🌳 Decision tree", "🔵 K-Means"])

with tab1:
    feature = st.selectbox("Numeric column", num, key="explore_col")
    window = st.slider("Rolling mean (days)", 1, 30, 7)
    ts = df[["Date", feature]].assign(rolling=df[feature].rolling(window, center=True).mean())
    st.plotly_chart(px.line(ts, x="Date", y=[feature, "rolling"], title=f"{feature} per day and {window}-day mean"), width="stretch")
    wd = df.assign(weekday=df.Date.dt.day_name())
    st.plotly_chart(px.box(wd, x="weekday", y=feature, category_orders={"weekday": WEEK}, title=f"{feature} by weekday"), width="stretch")
    st.dataframe(df.describe().T.round(2))

with tab2:
    default_t = num.index("CarAccidentsWithinCity") if "CarAccidentsWithinCity" in num else 0
    target_col = st.selectbox("Column that defines 'Severe' (above its median)", num, index=default_t, key="target_col")
    depth = st.slider("Max tree depth", 1, 8, 3)
    data = df.copy()
    data["Situation"] = np.where(data[target_col] > data[target_col].median(), "Severe", "Not Severe")
    counts = data["Situation"].value_counts()
    st.caption(f"Class balance: {counts.to_dict()}")
    if len(counts) < 2 or counts.min() < 10:
        st.error("This column is almost always the same value (e.g. zero), so the two classes are too unbalanced to train on. Pick another column.")
        st.stop()

    candidates = [c for c in num if c != target_col]
    leak = data[candidates].corrwith(data[target_col]).abs().sort_values(ascending=False)
    st.write("**Leakage check**: |correlation| of each predictor with the labelling column")
    st.dataframe(leak.rename("abs_corr").round(3).to_frame().T)
    flagged = leak[leak > 0.7].index.tolist()
    if flagged:
        st.warning(f"{flagged} are near-copies of '{target_col}'. Accuracy using them would be trivial; they are excluded by default.")
    X_cols = st.multiselect("Predictors", candidates, default=[c for c in candidates if c not in flagged])
    if st.checkbox("Add calendar features (weekday, month)", value=True):
        data["weekday_num"] = data.Date.dt.dayofweek; data["month"] = data.Date.dt.month
        X_cols = X_cols + ["weekday_num", "month"]
    if not X_cols:
        st.info("Choose at least one predictor."); st.stop()

    X_tr, X_te, y_tr, y_te = train_test_split(data[X_cols], data["Situation"], test_size=0.3, stratify=data["Situation"], random_state=42)
    model = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    acc_tr, acc_te = accuracy_score(y_tr, model.predict(X_tr)), accuracy_score(y_te, model.predict(X_te))
    baseline = y_te.value_counts(normalize=True).max()
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Train accuracy", f"{acc_tr:.3f}"); c2.metric("Test accuracy", f"{acc_te:.3f}")
    c3.metric("Baseline (majority class)", f"{baseline:.3f}"); c4.metric("Generalisation gap", f"{acc_tr - acc_te:+.3f}")
    if acc_te <= baseline + 0.02:
        st.info("The model barely beats guessing the majority class: these predictors carry little information about the label.")

    cm = confusion_matrix(y_te, model.predict(X_te), labels=model.classes_)
    l, r = st.columns(2)
    l.plotly_chart(px.imshow(cm, x=list(model.classes_), y=list(model.classes_), text_auto=True, color_continuous_scale="Blues",
                             labels=dict(x="predicted", y="actual"), title="Confusion matrix (test set)"), width="stretch")
    imp = pd.Series(model.feature_importances_, index=X_cols).sort_values()
    r.plotly_chart(px.bar(imp, orientation="h", title="Feature importance"), width="stretch")

    fig_tree, ax = plt.subplots(figsize=(min(4 + 3 * depth, 24), 2 + 1.5 * depth))
    plot_tree(model, feature_names=X_cols, class_names=list(model.classes_), filled=True, rounded=True, fontsize=8, ax=ax)
    st.pyplot(fig_tree)
    with st.expander("Text rules"):
        st.code(export_text(model, feature_names=X_cols))

with tab3:
    cols = st.multiselect("Features for clustering", num, default=num)
    if len(cols) < 2:
        st.info("Choose at least two features."); st.stop()
    Xs = StandardScaler().fit_transform(df[cols])
    ks = list(range(2, 9))
    sil = [silhouette_score(Xs, KMeans(k, n_init=10, random_state=42).fit_predict(Xs)) for k in ks]
    best = ks[int(np.argmax(sil))]
    st.plotly_chart(px.line(x=ks, y=sil, markers=True, labels=dict(x="k", y="silhouette"), title=f"Silhouette by k (best k = {best})"), width="stretch")
    k = st.slider("Number of clusters", 2, 8, best)
    labels = KMeans(k, n_init=10, random_state=42).fit_predict(Xs)
    pcs = PCA(2).fit_transform(Xs)
    st.plotly_chart(px.scatter(x=pcs[:, 0], y=pcs[:, 1], color=labels.astype(str), labels=dict(x="PC1", y="PC2", color="cluster"),
                               hover_name=df.Date.dt.strftime("%d %b"), title="Days clustered, shown in PCA space", opacity=0.7), width="stretch")
    st.write("Cluster profiles (mean per day):")
    st.dataframe(df[cols].assign(cluster=labels).groupby("cluster").mean().round(2).assign(days=np.bincount(labels)))
    st.write("Do clusters follow the calendar? (days per cluster × weekday)")
    st.dataframe(pd.crosstab(labels, df.Date.dt.day_name()).reindex(columns=WEEK, fill_value=0))

In [ ]:
at = AppTest.from_file(os.path.abspath("app_alkharj.py")).run(timeout=120)
assert not at.exception, at.exception
print("tabs:", [t.label for t in at.tabs])
tree_metrics = lambda: [m.value for m in at.metric][4:8]
print("depth 3 → train / test / baseline / gap:", tree_metrics())
at.slider[1].set_value(8).run(timeout=120)
print("depth 8 → train / test / baseline / gap:", tree_metrics())
at.selectbox[1].set_value("InjuredWithinCity").run(timeout=120)
print("leakage warning when labelling by InjuredWithinCity:", [w.value[:90] for w in at.warning])
if IN_COLAB:
    launch("app_alkharj.py", port=8503)

### ✏️ Exercise 1 — interpret
1. Choose `InjuredWithinCity` as the labelling column. Which predictor does the leakage check flag? Put it back into the predictors and compare test accuracy with and without it. Explain the difference.
2. With `CarAccidentsWithinCity` as the label, plot train and test accuracy against depth (a **validation curve**). Where does over-fitting start? Does the model ever clearly beat the majority-class baseline? What does that tell you about the original slide's result?
3. Is the weekday effect real? Test it (e.g. Kruskal–Wallis on accidents by weekday) and report the effect size, not only the p-value.
4. Do the K-Means clusters correspond to weekdays, months, or to rare high-casualty days? Use the crosstab and cluster profiles.

In [ ]:
# ✏️ Your code here (you can reuse the app's logic in plain Python)

In [ ]:
#@title 🔑 Solution — validation curve and weekday test (try first, then double-click to reveal) { display-mode: "form" }
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
d = pd.read_csv("2022.csv", encoding="utf-8-sig"); d["Date"] = pd.to_datetime(d["Date"], dayfirst=True)
target = "CarAccidentsWithinCity"
y = np.where(d[target] > d[target].median(), "Severe", "Not Severe")
X = d.select_dtypes("number").drop(columns=[target]).assign(weekday=d.Date.dt.dayofweek, month=d.Date.dt.month)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, stratify=y, random_state=42)
depths = range(1, 15); tr, te = [], []
for k in depths:
    m = DecisionTreeClassifier(max_depth=k, random_state=42).fit(Xtr, ytr)
    tr.append(m.score(Xtr, ytr)); te.append(m.score(Xte, yte))
plt.plot(depths, tr, "o-", label="train"); plt.plot(depths, te, "o-", label="test")
plt.axhline(pd.Series(yte).value_counts(normalize=True).max(), ls="--", c="grey", label="majority baseline")
plt.xlabel("max depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("Validation curve"); plt.show()

groups = [g[target].values for _, g in d.groupby(d.Date.dt.dayofweek)]
H, p = kruskal(*groups)
eps2 = (H - len(groups) + 1) / (len(d) - len(groups))       # epsilon-squared effect size
print(f"Kruskal–Wallis H = {H:.1f}, p = {p:.3g}, ε² = {eps2:.3f}")

## 8. Deploying beyond Colab
Colab links die when the runtime stops (≈ 90 min idle / 12 h max). For a permanent app:
1. Put `app.py`, the data file and a `requirements.txt` in a **GitHub repository**.
2. Go to **share.streamlit.io** → *New app* → pick the repo/branch/file → *Deploy*. Free for public apps.
3. Secrets (DB passwords, API keys) go in *App settings → Secrets*, read with `st.secrets[...]` — never commit them.

In [ ]:
%%writefile requirements.txt
streamlit
plotly
pandas
scikit-learn
matplotlib

In [ ]:
# Download the apps you built to your computer
if IN_COLAB:
    from google.colab import files
    for f in ["app_iris.py", "app_graali.py", "app_alkharj.py", "requirements.txt"]:
        files.download(f)

## 9. ✏️ Mini-lab (20–30 min)
Copy `app_iris.py` to `app_tips.py` and:
1. Replace iris with `px.data.tips()`.
2. Add a **range slider** for `total_bill` and a second chart (tip % by day).
3. Add one `st.metric` and a data preview.
4. Write **three** `AppTest` assertions (one must change a widget).

## 10. ✏️ Take-home project
Build a Streamlit dashboard for **your own research dataset**. Requirements:
* ≥ 2 filters, ≥ 2 coordinated charts, ≥ 1 KPI card, a data preview and CSV download.
* Caching for data loading; no hard-coded absolute paths.
* ≥ 5 `AppTest` tests, including an empty-filter edge case.
* If you include a model: a held-out evaluation, a leakage check and a sentence on its limitations.
* Deploy to Streamlit Community Cloud and submit the URL + repository.

## Quick check
1. What happens when a user moves a slider in Streamlit? What does that imply for expensive computations?
2. `st.cache_data` vs `st.cache_resource`: give one example of each.
3. Why is training accuracy of an unpruned decision tree uninformative?
4. Why do we need a tunnel or proxy to view a Streamlit app from Colab?

## References
* Streamlit docs — *App testing*: https://docs.streamlit.io/develop/api-reference/app-testing
* Kaufman, S. et al. (2012). *Leakage in Data Mining: Formulation, Detection, and Avoidance*. ACM TKDD.
* Rousseeuw, P. (1987). *Silhouettes: a graphical aid to the interpretation and validation of cluster analysis*. J. Comput. Appl. Math.

In [ ]:
# Run at the end of the session to stop servers and tunnels
stop_all()